# Notebook 1 — Train Original Model on Full CIFAR-10 (Paper Protocol)

**Paper:** *An Illusion of Unlearning? Assessing Machine Unlearning Through Internal Representations*  
Gao, Unal, Rangamani, Zhu — AISTATS 2026 · arXiv:2604.08271v1

**Purpose:**  
Train the **original model θ_o** on the full CIFAR-10 training set.  This checkpoint is the
shared starting point for all unlearning methods in Notebooks 2–4.

---

### Paper Protocol (§A.4 + Table 4)

| Setting | Value | Source |
|---|---|---|
| Architecture | ResNet-18 | §4 |
| Dataset | CIFAR-10 (50 000 train / 10 000 test) | §A.2 |
| Batch size | 128 | §A.4 |
| Epochs | up to 300 (early stop patience=50) | §A.4 |
| Optimiser | SGD momentum=0.9, WD=5×10⁻⁴, Nesterov | §A.4 |
| LR init | 5×10⁻² | §A.4 |
| LR schedule | 5-epoch linear warmup → cosine decay to 1×10⁻⁵ | §A.4 |
| Augmentation | RandomCrop(32, pad=4) + RandomHorizontalFlip | §A.4 |
| Val split | 10 % of training data for early stopping | §A.4 |
| Forget protocol | **Whole-class** — forget = 100 % of one class | §A.3 |

**Outputs saved:**
- `checkpoints/original_resnet18_cifar10.pt` — best-val checkpoint  
- `checkpoints/split_config.json` — class index lists for downstream notebooks

## Cell 0 — Install dependencies

In [ ]:
import subprocess, sys

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout: print(r.stdout[-3000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-1000:])
    return r.returncode

sh('pip install -q timm einops scikit-learn matplotlib seaborn pytorch-lightning torchmetrics')

## Cell 1 — Clone / update repository

In [ ]:
import os, subprocess

REPO_URL  = 'https://github.com/tiensinh2/CMF_Unlearning.git'
REPO_DIR  = '/kaggle/working/CMF_Unlearning'

if not os.path.isdir(REPO_DIR):
    subprocess.run(f'git clone {REPO_URL} {REPO_DIR}', shell=True, check=True)
else:
    subprocess.run(f'git -C {REPO_DIR} remote set-url origin {REPO_URL}', shell=True)
    subprocess.run(f'git -C {REPO_DIR} pull --ff-only', shell=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Working dir:', os.getcwd())

## Cell 2 — Configuration (paper §A.4 / Table 4)

In [ ]:
import os, random, math, json, time
import numpy as np
import torch

# ── Reproducibility ──────────────────────────────────────────
SEED = 0
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# ── Dataset ──────────────────────────────────────────────────
NUM_CLASSES   = 10          # CIFAR-10
DATASET       = 'cifar10'

# ── Training hyperparameters (paper §A.4 + Table 4) ──────────
BATCH_SIZE    = 128         # §A.4
MAX_EPOCHS    = 300         # §A.4: up to 300 epochs
PATIENCE      = 50          # §A.4: early stopping patience
LR_INIT       = 1e-2        # Table 4 line 2308: lr=0.01  (§A.4 text says 5e-2 — Table 4 takes precedence)
LR_MIN        = 1e-5        # §A.4: cosine min LR
WARMUP_EPOCHS = 5           # §A.4: 5-epoch linear warmup
MOMENTUM      = 0.9         # §A.4
WEIGHT_DECAY  = 5e-4        # §A.4: WD = 5×10⁻⁴
NESTEROV      = True        # §A.4
VAL_RATIO     = 0.1         # §A.4: 10 % of train → validation

# ── Paths ─────────────────────────────────────────────────────
CKPT_DIR      = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

CKPT_ORIG     = os.path.join(CKPT_DIR, 'original_resnet18_cifar10.pt')
SPLIT_CFG     = os.path.join(CKPT_DIR, 'split_config.json')

print(f'Checkpoint will be saved to: {CKPT_ORIG}')

## Cell 3 — Load CIFAR-10 with paper augmentation

In [ ]:
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset, random_split

# Paper §A.4: RandomCrop(32, pad=4) + RandomHorizontalFlip + Normalize
# CIFAR-10 standard channel statistics
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

train_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# Kaggle: /kaggle/input is read-only; try pre-mounted dataset first,
# then fall back to downloading into the writable /kaggle/working/data.
import os as _os
_KAGGLE_INPUT = '/kaggle/input'
_WRITABLE     = '/kaggle/working/data'

def _find_cifar10_root():
    for candidate in [
        _os.path.join(_KAGGLE_INPUT, 'cifar-10-batches-py'),
        _KAGGLE_INPUT,
    ]:
        if _os.path.exists(_os.path.join(candidate, 'cifar-10-batches-py')):
            return candidate, False   # (root, need_download)
    return _WRITABLE, True

DATA_ROOT, NEED_DOWNLOAD = _find_cifar10_root()
_os.makedirs(DATA_ROOT, exist_ok=True)
print(f'DATA_ROOT={DATA_ROOT}  download={NEED_DOWNLOAD}')

full_train_dataset = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=True,
                                                   download=NEED_DOWNLOAD, transform=train_transform)
full_test_dataset  = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=False,
                                                   download=NEED_DOWNLOAD, transform=test_transform)

# § A.4: 10 % of training data → validation (for early stopping)
n_total  = len(full_train_dataset)   # 50 000
n_val    = int(n_total * VAL_RATIO)  # 5 000
n_train  = n_total - n_val           # 45 000

gen = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(full_train_dataset,
                                           [n_train, n_val], generator=gen)

# val set should use test_transform (no augmentation)
val_dataset_eval = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=True,
                                                 download=NEED_DOWNLOAD, transform=test_transform)
val_dataset_eval = Subset(val_dataset_eval, val_dataset.indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset_eval, batch_size=256,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(full_test_dataset, batch_size=256,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset):,}  Val: {len(val_dataset):,}  Test: {len(full_test_dataset):,}')

## Cell 4 — Build ResNet-18 (paper §4)

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

from models.resnet import ResNet18

model = ResNet18(num_classes=NUM_CLASSES, dataset='cifar10').to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'ResNet-18  |  Parameters: {total_params:,}')

# Smoke-test forward pass
with torch.no_grad():
    dummy = torch.randn(2, 3, 32, 32).to(DEVICE)
    out   = model(dummy)
print(f'Forward pass OK — output shape: {out.shape}')   # expect [2, 10]

## Cell 5 — Optimizer and LR scheduler (paper §A.4)

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

optimizer = optim.SGD(
    model.parameters(),
    lr=LR_INIT,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    nesterov=NESTEROV,
)

# §A.4: 5-epoch linear warmup then cosine decay to LR_MIN
warmup_scheduler = LinearLR(
    optimizer,
    start_factor=1e-2,   # LR_INIT * 1e-2 = 1e-4 at epoch 0
    end_factor=1.0,
    total_iters=WARMUP_EPOCHS,
)
cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS - WARMUP_EPOCHS,
    eta_min=LR_MIN,
)
scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[WARMUP_EPOCHS],
)

print('Optimizer:  SGD  lr={:.2e}  momentum={:.1f}  wd={:.1e}  nesterov={}'.format(
      LR_INIT, MOMENTUM, WEIGHT_DECAY, NESTEROV))
print('Scheduler:  LinearLR ({}ep warmup) → CosineAnnealingLR → {:.0e}'.format(
      WARMUP_EPOCHS, LR_MIN))

## Cell 6 — Training loop with early stopping

In [ ]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss()

def evaluate(model, loader, device):
    """Return accuracy (0–100) on loader."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            # model outputs log-softmax; argmax is same as softmax argmax
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total   += y.size(0)
    return 100.0 * correct / max(1, total)

def train_one_epoch(model, loader, optimizer, criterion, device):
    """Run one training epoch. Returns (avg_loss, accuracy)."""
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out  = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        bs        = y.size(0)
        loss_sum += loss.item() * bs
        correct  += (out.argmax(1) == y).sum().item()
        total    += bs
    return loss_sum / max(1, total), 100.0 * correct / max(1, total)

# ── Training ──────────────────────────────────────────────────
best_val_acc   = -1.0
best_state     = None
no_improve_cnt = 0
history        = []

print('Epoch | Train Loss | Train Acc | Val Acc |  LR')
print('------+-----------+----------+---------+--------')

t0 = time.time()
for epoch in range(1, MAX_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_acc         = evaluate(model, val_loader, DEVICE)
    lr_now          = optimizer.param_groups[0]['lr']
    scheduler.step()

    history.append(dict(epoch=epoch, tr_loss=tr_loss, tr_acc=tr_acc,
                        val_acc=val_acc, lr=lr_now))

    if epoch % 10 == 0 or epoch <= 5:
        print(f'  {epoch:3d}  | {tr_loss:.4f}     | {tr_acc:.2f}%   | {val_acc:.2f}%  | {lr_now:.2e}')

    # Early stopping (paper §A.4: patience=50)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve_cnt = 0
    else:
        no_improve_cnt += 1
        if no_improve_cnt >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}  (best val={best_val_acc:.2f}%)')
            break

elapsed = time.time() - t0
print(f'\nTraining done in {elapsed/60:.1f} min.  Best val acc = {best_val_acc:.2f}%')

# Restore best weights
model.load_state_dict(best_state)

## Cell 7 — Evaluate on test set

In [ ]:
test_acc = evaluate(model, test_loader, DEVICE)
print(f'Test accuracy (full dataset): {test_acc:.2f}%')
print('Paper Table 1 reference (Original, Output): ~93.98 %  (CIFAR-10)')

## Cell 8 — Save checkpoint + split config

In [ ]:
import torch
import json, os
import torchvision

# ── Save model checkpoint ────────────────────────────────────
torch.save({
    'model_state_dict' : model.state_dict(),
    'test_acc'         : test_acc,
    'best_val_acc'     : best_val_acc,
    'seed'             : SEED,
    'epoch_stopped'    : len(history),
    'hparams': {
        'lr_init'      : LR_INIT,
        'batch_size'   : BATCH_SIZE,
        'momentum'     : MOMENTUM,
        'weight_decay' : WEIGHT_DECAY,
        'warmup_epochs': WARMUP_EPOCHS,
        'max_epochs'   : MAX_EPOCHS,
        'patience'     : PATIENCE,
    },
    'paper': 'arXiv:2604.08271v1 §A.4 + Table 4',
}, CKPT_ORIG)
print(f'Saved checkpoint → {CKPT_ORIG}')

# ── Build and save per-class index lists ─────────────────────
# Paper §A.3: whole-class protocol — forget = ALL samples of one class.
# We save the index lists so Notebooks 2–4 can reproduce splits deterministically.

full_train_for_idx = torchvision.datasets.CIFAR10(
    root=DATA_ROOT, train=True, download=NEED_DOWNLOAD,
    transform=torchvision.transforms.ToTensor())

targets = np.array(full_train_for_idx.targets)  # shape [50000]

# TRAIN_CLASS_IDX[c] = list of all training indices for class c
TRAIN_CLASS_IDX = {c: np.where(targets == c)[0].tolist() for c in range(NUM_CLASSES)}

# TEST_CLASS_IDX[c] = list of all test indices for class c
test_targets = np.array(full_test_dataset.targets)
TEST_CLASS_IDX = {c: np.where(test_targets == c)[0].tolist() for c in range(NUM_CLASSES)}

# Verify sizes: CIFAR-10 has 5 000 train / 1 000 test per class
for c in range(NUM_CLASSES):
    assert len(TRAIN_CLASS_IDX[c]) == 5000, f'class {c} train size mismatch'
    assert len(TEST_CLASS_IDX[c])  == 1000, f'class {c} test size mismatch'

# Unlearning scenarios from paper §A.3
SINGLE_CLASS_FORGET = [[c] for c in range(10)]       # sweep all 10 classes
MULTI_CLASS_FORGET  = [
    [0, 1, 2],
    [3, 4, 5],
    [6, 7, 8],
    [0, 5, 9],
    [2, 4, 8],
]

split_config = {
    'dataset'               : DATASET,
    'num_classes'           : NUM_CLASSES,
    'seed'                  : SEED,
    'protocol'              : 'whole_class',   # paper §A.3
    'TRAIN_CLASS_IDX'       : TRAIN_CLASS_IDX,
    'TEST_CLASS_IDX'        : TEST_CLASS_IDX,
    'SINGLE_CLASS_FORGET'   : SINGLE_CLASS_FORGET,
    'MULTI_CLASS_FORGET'    : MULTI_CLASS_FORGET,
    'paper'                 : 'arXiv:2604.08271v1 §A.3',
}

with open(SPLIT_CFG, 'w') as f:
    json.dump(split_config, f, indent=2)

print(f'Saved split config → {SPLIT_CFG}')
print(f'Training samples per class: {[len(v) for v in TRAIN_CLASS_IDX.values()]}')
print(f'Test     samples per class: {[len(v) for v in TEST_CLASS_IDX.values()]}')

## Cell 9 — Plot learning curves

In [ ]:
import matplotlib.pyplot as plt

epochs_run = [h['epoch']   for h in history]
tr_accs    = [h['tr_acc']  for h in history]
val_accs   = [h['val_acc'] for h in history]
lrs        = [h['lr']      for h in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_run, tr_accs,  label='Train acc')
axes[0].plot(epochs_run, val_accs, label='Val acc')
axes[0].axhline(test_acc, color='red', linestyle='--', label=f'Test acc ({test_acc:.1f}%)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Training / Validation Accuracy')
axes[0].legend()

axes[1].plot(epochs_run, lrs)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('LR')
axes[1].set_title('Learning Rate Schedule')
axes[1].set_yscale('log')

plt.suptitle('Notebook 1 — Original ResNet-18 on CIFAR-10  (paper §A.4)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, 'nb1_training_curves.png'), dpi=100)
plt.show()
print('Plot saved.')

## Cell 10 — Summary

In [ ]:
print('=' * 55)
print('  Notebook 1 — Summary')
print('=' * 55)
print(f'  Architecture  : ResNet-18  (CIFAR-10 variant)')
print(f'  Epochs run    : {len(history)}')
print(f'  Best val acc  : {best_val_acc:.2f}%')
print(f'  Test  acc     : {test_acc:.2f}%   (paper ref: ~93.98%)')
print(f'  Checkpoint    : {CKPT_ORIG}')
print(f'  Split config  : {SPLIT_CFG}')
print()
print('  Paper protocol (§A.3):')
print('  • Forget = whole class (100% of one class)')
print('  • Retain = all other 9 classes')
print('  • Single-class forget: sweep all 10 classes')
print('  • Multi-class forget : 5 representative 3-class sets')
print()
print('  ➜  Run Notebook 2 (Oracle Retrain) next.')
print('=' * 55)